In [3]:
import warnings
warnings.filterwarnings("ignore")      # oculta avisos técnicos para que la pantalla quede limpia

import numpy as np                     # cálculo numérico
import pandas as pd                    # tablas de datos
import matplotlib.pyplot as plt        # gráficas
import kagglehub                       # descarga datasets de Kaggle

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, ConfusionMatrixDisplay, confusion_matrix,
                             accuracy_score, recall_score, precision_score, f1_score)

---
## 1 · Los datos

**Heart Failure Prediction Dataset** (fedesoriano, Kaggle): 918 pacientes de cinco cohortes cardiológicas.

| Variable | Significado clínico |
|---|---|
| `Age` | Edad (años) |
| `Sex` | Sexo (M / F) |
| `ChestPainType` | Dolor torácico: TA angina típica · ATA atípica · NAP no anginoso · ASY asintomático |
| `RestingBP` | Presión arterial sistólica en reposo (mm Hg) |
| `Cholesterol` | Colesterol sérico (mg/dL) |
| `FastingBS` | Glucosa en ayuno > 120 mg/dL (1 = sí) |
| `RestingECG` | ECG en reposo: Normal · ST (anomalía ST-T) · LVH (hipertrofia ventricular izquierda) |
| `MaxHR` | Frecuencia cardiaca máxima en prueba de esfuerzo |
| `ExerciseAngina` | Angina inducida por el ejercicio (Y / N) |
| `Oldpeak` | Depresión del segmento ST con el esfuerzo (mm) |
| `ST_Slope` | Pendiente del ST en el pico del esfuerzo: Up · Flat · Down |
| **`HeartDisease`** | **Lo que queremos predecir: 1 = con enfermedad cardiaca · 0 = sin enfermedad** |

In [4]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
df = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS,
                            "fedesoriano/heart-failure-prediction", "heart.csv")

In [5]:
df.head

<bound method NDFrame.head of      Age Sex ChestPainType  RestingBP  Cholesterol  FastingBS RestingECG  \
0     40   M           ATA        140          289          0     Normal   
1     49   F           NAP        160          180          0     Normal   
2     37   M           ATA        130          283          0         ST   
3     48   F           ASY        138          214          0     Normal   
4     54   M           NAP        150          195          0     Normal   
..   ...  ..           ...        ...          ...        ...        ...   
913   45   M            TA        110          264          0     Normal   
914   68   M           ASY        144          193          1     Normal   
915   57   M           ASY        130          131          0     Normal   
916   57   F           ATA        130          236          0        LVH   
917   38   M           NAP        138          175          0     Normal   

     MaxHR ExerciseAngina  Oldpeak ST_Slope  HeartDisease

In [6]:
#Imprimuiendo renglones y columnnas del dataframe
df.shape

(918, 12)

#Preparando los datos


In [7]:
#Imprimiendo tipo de datos
df.dtypes

Age                 int64
Sex                   str
ChestPainType         str
RestingBP           int64
Cholesterol         int64
FastingBS           int64
RestingECG            str
MaxHR               int64
ExerciseAngina        str
Oldpeak           float64
ST_Slope              str
HeartDisease        int64
dtype: object

In [8]:
#Comprobando si existen datos nulos
df.isna().sum

<bound method DataFrame.sum of        Age    Sex  ChestPainType  RestingBP  Cholesterol  FastingBS  \
0    False  False          False      False        False      False   
1    False  False          False      False        False      False   
2    False  False          False      False        False      False   
3    False  False          False      False        False      False   
4    False  False          False      False        False      False   
..     ...    ...            ...        ...          ...        ...   
913  False  False          False      False        False      False   
914  False  False          False      False        False      False   
915  False  False          False      False        False      False   
916  False  False          False      False        False      False   
917  False  False          False      False        False      False   

     RestingECG  MaxHR  ExerciseAngina  Oldpeak  ST_Slope  HeartDisease  
0         False  False           False    

In [9]:
#Explorando los datos numericos
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Age,918.0,53.510893,9.432617,28.0,47.00,54.0,60.0,77.0
RestingBP,918.0,132.396514,18.514154,0.0,120.00,130.0,140.0,200.0
Cholesterol,918.0,198.799564,109.384145,0.0,173.25,223.0,267.0,603.0
FastingBS,918.0,0.233115,0.423046,0.0,0.00,0.0,0.0,1.0
MaxHR,918.0,136.809368,25.460334,60.0,120.00,138.0,156.0,202.0
Oldpeak,918.0,0.887364,1.066570,-2.6,0.00,0.6,1.5,6.2
HeartDisease,918.0,0.553377,0.497414,0.0,0.00,1.0,1.0,1.0


In [11]:
#Corrigiendo los valores numericos en 0 que no hacen sentido
df['Cholesterol']=df['Cholesterol'].replace(0, np.nan)
df['RestingBP']=df['RestingBP'].replace(0, np.nan)

# Dividiendo los datos y aplicando one hot

In [12]:
#Separando los datos en x y y
y= df['HeartDisease']
x=df.drop(columns=['HeartDisease'])

In [13]:
x

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope
0,40,M,ATA,140.0,289.0,0,Normal,172,N,0.0,Up
1,49,F,NAP,160.0,180.0,0,Normal,156,N,1.0,Flat
2,37,M,ATA,130.0,283.0,0,ST,98,N,0.0,Up
3,48,F,ASY,138.0,214.0,0,Normal,108,Y,1.5,Flat
4,54,M,NAP,150.0,195.0,0,Normal,122,N,0.0,Up
...,...,...,...,...,...,...,...,...,...,...,...
913,45,M,TA,110.0,264.0,0,Normal,132,N,1.2,Flat
914,68,M,ASY,144.0,193.0,1,Normal,141,N,3.4,Flat
915,57,M,ASY,130.0,131.0,0,Normal,115,Y,1.2,Flat
916,57,F,ATA,130.0,236.0,0,LVH,174,N,0.0,Flat


In [22]:
#Definiendo columnas de tipo caracter o string
string_columns= ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina','ST_Slope']

In [23]:
#Iterando sobre las columnas string
for column in string_columns:
    print(x[column].value_counts())
    print('\n')

Sex
M    725
F    193
Name: count, dtype: int64


ChestPainType
ASY    496
NAP    203
ATA    173
TA      46
Name: count, dtype: int64


RestingECG
Normal    552
LVH       188
ST        178
Name: count, dtype: int64


ExerciseAngina
N    547
Y    371
Name: count, dtype: int64


ST_Slope
Flat    460
Up      395
Down     63
Name: count, dtype: int64




In [24]:
#Imprimiendo los datos antes de aplicar el one hot
print(x.head())
print(x.shape)


   Age Sex ChestPainType  RestingBP  Cholesterol  FastingBS RestingECG  MaxHR  \
0   40   M           ATA      140.0        289.0          0     Normal    172   
1   49   F           NAP      160.0        180.0          0     Normal    156   
2   37   M           ATA      130.0        283.0          0         ST     98   
3   48   F           ASY      138.0        214.0          0     Normal    108   
4   54   M           NAP      150.0        195.0          0     Normal    122   

  ExerciseAngina  Oldpeak ST_Slope  
0              N      0.0       Up  
1              N      1.0     Flat  
2              N      0.0       Up  
3              Y      1.5     Flat  
4              N      0.0       Up  
(918, 11)


In [2]:
#Aplicando one-hot a las columnas string
x = pd.get_dummies(x, drop_first=True, dtype = int)
print(x.head())
print(x.shape)

NameError: name 'pd' is not defined

# Haciendo el split de los datos

In [ ]:
#Imprimiendo la proporcion de la variable objetivo
print(df['HeartDisease'])

In [ ]:
x_train, x_test, y_train 